# Atividade Prática 02 — Estruturando uma Camada de Lotes

**Disciplina:** [12517] Arquitetura de Grandes Volumes de Dados  
**Prof.:** Alexandre Seidy Ioshisaqui

**Membros:**

* Giulia Monteiro Garrido (RA: 24010281)
* Thomaz Dacorso (RA: 24012310)
* Victor França (RA: 24010801)
* Vitor Furuta (RA: 24008775)

## Contexto e Objetivo

Na Etapa 1, exploramos o dataset **Amazon Sales Dataset** (Kaggle: `aliiihussain/amazon-sales-dataset`), que registra **50.000 eventos de venda** ocorridos entre janeiro de 2022 e dezembro de 2023. Cada linha representa um pedido individual, contendo data, produto, categoria, região do cliente, método de pagamento, preço, desconto e avaliação.

As perguntas investigadas na Etapa 1 foram:
1. Qual mês gerou maior receita?
2. Qual categoria de produto vende mais e gera mais receita?
3. Qual região tem o maior ticket médio?
4. Qual método de pagamento é mais utilizado?
5. Quais categorias oferecem os maiores descontos?

Nesta Etapa 2, convertemos toda essa estrutura para uma **Camada de Lotes**, composta por:
- **Particionamento** dos dados brutos por ano e mês;
- **Vistas de lote** que pré-computam as agregações das consultas acima;
- **Arquivos de gerenciamento** para rastreamento de processamento e controle de qualidade.

## 1. Imports e Constantes

In [ ]:
import os
import csv
import pandas as pd
import numpy as np
from datetime import datetime
import kagglehub

# Diretórios da Camada de Lote
DADOS_DIR         = "dados_brutos"
VISTAS_DIR        = "vistas_lote"
GERENCIAMENTO_DIR = "gerenciamento"

os.makedirs(DADOS_DIR, exist_ok=True)
os.makedirs(VISTAS_DIR, exist_ok=True)
os.makedirs(GERENCIAMENTO_DIR, exist_ok=True)

print("Diretórios criados com sucesso.")

## 2. Carregamento dos Dados Brutos

In [ ]:
# Download do dataset
path = kagglehub.dataset_download("aliiihussain/amazon-sales-dataset")
print("Path to dataset files:", path)

In [ ]:
name_files = os.listdir(path)
raw_path = os.path.join(path, name_files[0])

df = pd.read_csv(raw_path)
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dataset carregado: {len(df)} linhas, {df.shape[1]} colunas")
df.info()

## 3. Validação e Quarentena

### Decisão de projeto: validação antes do particionamento

Optou-se por realizar a etapa de validação e quarentena **antes** do particionamento. Essa decisão tem como objetivo garantir que apenas registros íntegros sejam escritos nas partições de dados brutos — evitando propagar erros para as vistas de lote e para os resultados das consultas.

Os critérios de rejeição aplicados a cada linha são:

| Critério | Motivo registrado |
|---|---|
| Qualquer campo nulo | `campo_nulo` |
| `price`, `total_revenue` ou `discounted_price` ≤ 0 | `valor_financeiro_invalido` |
| `quantity_sold` < 1 | `quantidade_invalida` |
| `rating` fora do intervalo [1.0, 5.0] | `rating_fora_do_intervalo` |
| `discount_percent` fora do intervalo [0, 100] | `desconto_invalido` |

Linhas rejeitadas são salvas em `gerenciamento/quarentena_linhas_brutas.csv` para rastreabilidade, junto com o arquivo de resumo `quarentena_linhas_por_arquivo.csv` que registra quantas linhas foram rejeitadas por arquivo de origem.

In [ ]:
QUARENTENA_LINHAS_PATH = os.path.join(GERENCIAMENTO_DIR, "quarentena_linhas_brutas.csv")
QUARENTENA_RESUMO_PATH = os.path.join(GERENCIAMENTO_DIR, "quarentena_linhas_por_arquivo.csv")

def validar_linha(row):
    """Retorna None se a linha é válida, ou uma string descrevendo o motivo de rejeição."""
    if row.isnull().any():
        return "campo_nulo"
    if row["price"] <= 0 or row["total_revenue"] <= 0 or row["discounted_price"] <= 0:
        return "valor_financeiro_invalido"
    if row["quantity_sold"] < 1:
        return "quantidade_invalida"
    if not (1.0 <= row["rating"] <= 5.0):
        return "rating_fora_do_intervalo"
    if not (0 <= row["discount_percent"] <= 100):
        return "desconto_invalido"
    return None

# Aplica validação
motivos = df.apply(validar_linha, axis=1)
mask_validas = motivos.isna()

df_valido     = df[mask_validas].copy()
df_quarentena = df[~mask_validas].copy()
df_quarentena["motivo_quarentena"] = motivos[~mask_validas]
df_quarentena["arquivo_origem"]    = os.path.basename(raw_path)

# Salva linhas em quarentena
df_quarentena.to_csv(QUARENTENA_LINHAS_PATH, index=False)

# Resumo de quarentena por arquivo
resumo_quarentena = pd.DataFrame([{
    "arquivo":           os.path.basename(raw_path),
    "linhas_totais":     len(df),
    "linhas_validas":    len(df_valido),
    "linhas_quarentena": len(df_quarentena),
    "timestamp":         datetime.now().isoformat()
}])
resumo_quarentena.to_csv(QUARENTENA_RESUMO_PATH, index=False)

print(f"Total de linhas     : {len(df)}")
print(f"Linhas válidas      : {len(df_valido)}")
print(f"Linhas em quarentena: {len(df_quarentena)}")
if len(df_quarentena) > 0:
    print("\nMotivos de quarentena:")
    print(df_quarentena["motivo_quarentena"].value_counts())

## 4. Particionamento dos Dados Brutos

### Decisão de projeto: granularidade mensal

O dataset cobre **24 meses** de eventos (janeiro/2022 a dezembro/2023), com aproximadamente **2.000 registros por mês**.

A granularidade de partição escolhida foi **ano/mês** (`dados_brutos/{ano}/{mm}/dados.csv`), pelos seguintes motivos:

- **Tamanho adequado por partição:** ~2.000 linhas por partição é um volume razoável — nem pequeno demais (o que geraria overhead excessivo de arquivos) nem grande demais (o que dificultaria reprocessamentos parciais).
- **Alinhamento com as consultas:** as principais análises da Etapa 1 usam agregações mensais (ex.: receita total por mês). Partições mensais permitem que uma consulta por período específico leia apenas os arquivos relevantes, sem varrer o dataset completo.
- **Granularidade diária seria excessiva:** com ~67 registros/dia, criar uma partição por dia resultaria em 730 arquivos muito pequenos, sem benefício prático para as consultas definidas.
- **Granularidade anual seria insuficiente:** apenas 2 partições (2022 e 2023), cada uma com ~25.000 linhas, tornaria o reprocessamento custoso e impediria consultas eficientes por período menor que um ano.

O log de processamento registra cada partição gerada com status e timestamp, permitindo auditoria e reprocessamento seletivo.

In [ ]:
LOG_PATH = os.path.join(GERENCIAMENTO_DIR, "log_processamento.csv")

# Inicializa o log de processamento
log_colunas   = ["arquivo", "ano", "mes", "linhas_processadas", "status", "timestamp"]
log_registros = []

years  = sorted(df_valido["order_date"].dt.year.unique())
months = sorted(df_valido["order_date"].dt.month.unique())

for year in years:
    for month in months:
        folder  = os.path.join(DADOS_DIR, str(year), f"{month:02d}")
        os.makedirs(folder, exist_ok=True)
        arquivo = os.path.join(folder, "dados.csv")

        df_part = df_valido[
            (df_valido["order_date"].dt.year  == year) &
            (df_valido["order_date"].dt.month == month)
        ]

        try:
            df_part.to_csv(arquivo, index=False)
            status = "ok"
        except Exception as e:
            status = f"erro: {e}"

        log_registros.append({
            "arquivo":            arquivo,
            "ano":                year,
            "mes":                month,
            "linhas_processadas": len(df_part),
            "status":             status,
            "timestamp":          datetime.now().isoformat()
        })

# Salva log de processamento
pd.DataFrame(log_registros, columns=log_colunas).to_csv(LOG_PATH, index=False)

print(f"Particionamento concluído: {len(years)} anos × {len(months)} meses = {len(log_registros)} partições")
print(f"Log salvo em: {LOG_PATH}")

## 5. Construção das Vistas de Lote

### Decisão de projeto: quais vistas construir e por quê

Cada vista de lote pré-computa uma agregação específica e é armazenada como um arquivo CSV indexado. As vistas foram definidas diretamente a partir das **perguntas investigadas na Etapa 1**, de forma que cada consulta relevante possa ser respondida lendo apenas o arquivo de vista correspondente — sem precisar reprocessar as partições brutas.

| Vista | Arquivo gerado | Pergunta da Etapa 1 que responde |
|---|---|---|
| Receita total por mês | `receita_por_mes.csv` | Qual mês gerou maior receita? |
| Vendas e receita por categoria | `vendas_por_categoria.csv` | Qual categoria vende mais e gera mais receita? |
| Vendas por região | `vendas_por_regiao.csv` | Qual região tem maior ticket médio? |
| Vendas por método de pagamento | `vendas_por_pagamento.csv` | Qual método de pagamento é mais usado? |
| Desconto médio por categoria | `desconto_por_categoria.csv` | Quais categorias têm maiores descontos? |
| Vendas por dia (série temporal) | `vendas_por_dia.csv` | Como a receita evolui ao longo do tempo? |

Cada vista produz um arquivo com **um valor agregado por linha**, indexado por uma dimensão (período, categoria, região ou método de pagamento). Esse formato segue o padrão de Camada de Lote descrito no enunciado.

In [ ]:
def construir_vistas_de_lote(df):
    """Constrói todas as vistas de lote a partir dos dados válidos e as salva em CSV."""
    os.makedirs(VISTAS_DIR, exist_ok=True)

    # --- Vista 1: Receita total por mês ---
    # Responde: qual mês gerou maior receita?
    vista_receita_mes = (
        df.groupby(df["order_date"].dt.to_period("M"))["total_revenue"]
        .sum()
        .reset_index()
        .rename(columns={"order_date": "ano_mes", "total_revenue": "receita_total"})
    )
    vista_receita_mes["ano_mes"] = vista_receita_mes["ano_mes"].astype(str)
    vista_receita_mes.to_csv(os.path.join(VISTAS_DIR, "receita_por_mes.csv"), index=False)

    # --- Vista 2: Contagem de vendas e receita por categoria ---
    # Responde: qual categoria vende mais e gera mais receita?
    vista_categoria = (
        df.groupby("product_category")
        .agg(
            total_vendas =("order_id",       "count"),
            receita_total=("total_revenue",   "sum"),
            preco_medio  =("price",           "mean"),
            rating_medio =("rating",          "mean"),
        )
        .reset_index()
        .round(2)
    )
    vista_categoria.to_csv(os.path.join(VISTAS_DIR, "vendas_por_categoria.csv"), index=False)

    # --- Vista 3: Vendas e ticket médio por região ---
    # Responde: qual região tem maior ticket médio?
    vista_regiao = (
        df.groupby("customer_region")
        .agg(
            total_vendas =("order_id",      "count"),
            receita_total=("total_revenue", "sum"),
            ticket_medio =("total_revenue", "mean"),
        )
        .reset_index()
        .round(2)
    )
    vista_regiao.to_csv(os.path.join(VISTAS_DIR, "vendas_por_regiao.csv"), index=False)

    # --- Vista 4: Vendas por método de pagamento ---
    # Responde: qual método de pagamento é mais utilizado?
    vista_pagamento = (
        df.groupby("payment_method")
        .agg(
            total_vendas =("order_id",      "count"),
            receita_total=("total_revenue", "sum"),
        )
        .reset_index()
        .round(2)
    )
    vista_pagamento.to_csv(os.path.join(VISTAS_DIR, "vendas_por_pagamento.csv"), index=False)

    # --- Vista 5: Desconto médio por categoria ---
    # Responde: quais categorias oferecem os maiores descontos?
    vista_desconto = (
        df.groupby("product_category")
        .agg(
            desconto_medio  =("discount_percent",  "mean"),
            preco_desc_medio=("discounted_price",   "mean"),
            qtd_media       =("quantity_sold",      "mean"),
        )
        .reset_index()
        .round(2)
    )
    vista_desconto.to_csv(os.path.join(VISTAS_DIR, "desconto_por_categoria.csv"), index=False)

    # --- Vista 6: Contagem de vendas por dia (série temporal) ---
    # Responde: como a receita e o volume de vendas evoluem ao longo do tempo?
    vista_dia = (
        df.groupby(df["order_date"].dt.date)
        .agg(
            total_vendas =("order_id",      "count"),
            receita_total=("total_revenue", "sum"),
        )
        .reset_index()
        .rename(columns={"order_date": "data"})
        .round(2)
    )
    vista_dia.to_csv(os.path.join(VISTAS_DIR, "vendas_por_dia.csv"), index=False)

    print(f"6 vistas de lote construídas em '{VISTAS_DIR}/'")
    return {
        "receita_por_mes":        vista_receita_mes,
        "vendas_por_categoria":   vista_categoria,
        "vendas_por_regiao":      vista_regiao,
        "vendas_por_pagamento":   vista_pagamento,
        "desconto_por_categoria": vista_desconto,
        "vendas_por_dia":         vista_dia,
    }

vistas = construir_vistas_de_lote(df_valido)

## 6. Consultas sobre as Vistas de Lote

As consultas a seguir reproduzem os resultados obtidos na Etapa 1, mas agora lendo **exclusivamente os arquivos de vista de lote** — sem acessar os dados brutos ou o DataFrame original. Isso demonstra a principal vantagem da Camada de Lotes: as agregações já estão pré-computadas, tornando as consultas muito mais rápidas e independentes do volume bruto de dados.

Em produção, um novo lote de dados seria processado periodicamente (ex.: diariamente ou mensalmente), as vistas seriam reconstruídas ou incrementadas, e as consultas continuariam funcionando da mesma forma.

In [ ]:
# Leitura das vistas do disco (simulando consulta em produção)
v_receita   = pd.read_csv(os.path.join(VISTAS_DIR, "receita_por_mes.csv"))
v_categoria = pd.read_csv(os.path.join(VISTAS_DIR, "vendas_por_categoria.csv"))
v_regiao    = pd.read_csv(os.path.join(VISTAS_DIR, "vendas_por_regiao.csv"))
v_pagamento = pd.read_csv(os.path.join(VISTAS_DIR, "vendas_por_pagamento.csv"))
v_desconto  = pd.read_csv(os.path.join(VISTAS_DIR, "desconto_por_categoria.csv"))
v_dia       = pd.read_csv(os.path.join(VISTAS_DIR, "vendas_por_dia.csv"))

# Consulta 1: Top 3 meses com maior receita
# Fonte: receita_por_mes.csv | Coluna de índice: ano_mes | Agregação: soma da receita
print("=== Consulta 1: Top 3 meses por receita ===")
print(v_receita.nlargest(3, "receita_total").to_string(index=False))

# Consulta 2: Categoria com maior número de vendas
# Fonte: vendas_por_categoria.csv | Coluna de índice: product_category | Agregação: contagem e soma
print("\n=== Consulta 2: Vendas por categoria ===")
print(v_categoria.sort_values("total_vendas", ascending=False).to_string(index=False))

# Consulta 3: Região com maior ticket médio
# Fonte: vendas_por_regiao.csv | Coluna de índice: customer_region | Agregação: média da receita por pedido
print("\n=== Consulta 3: Ticket médio por região ===")
print(v_regiao.sort_values("ticket_medio", ascending=False).to_string(index=False))

# Consulta 4: Método de pagamento mais usado
# Fonte: vendas_por_pagamento.csv | Coluna de índice: payment_method | Agregação: contagem
print("\n=== Consulta 4: Vendas por método de pagamento ===")
print(v_pagamento.sort_values("total_vendas", ascending=False).to_string(index=False))

# Consulta 5: Categoria com maior desconto médio
# Fonte: desconto_por_categoria.csv | Coluna de índice: product_category | Agregação: média do desconto
print("\n=== Consulta 5: Desconto médio por categoria ===")
print(v_desconto.sort_values("desconto_medio", ascending=False).to_string(index=False))

## 7. Verificação dos Arquivos de Gerenciamento

### Rotinas de gerenciamento implementadas

A Camada de Lotes conta com três arquivos de gerenciamento, cada um com uma função específica:

**a) `log_processamento.csv` — controle de estado do processamento**  
Registra cada partição gerada com: caminho do arquivo, ano, mês, quantidade de linhas processadas, status (`ok` ou mensagem de erro) e timestamp. Permite verificar se todas as partições foram geradas com sucesso e identificar falhas sem precisar reprocessar tudo.

**b) `quarentena_linhas_brutas.csv` — depuração: linhas rejeitadas**  
Armazena cada linha que falhou na validação, incluindo todos os seus campos originais, o motivo da rejeição e o arquivo de origem. Permite inspecionar manualmente os registros problemáticos e, se necessário, corrigi-los e reinserí-los no pipeline.

**c) `quarentena_linhas_por_arquivo.csv` — depuração: resumo por arquivo**  
Para cada arquivo de entrada processado, registra o total de linhas, quantas foram aceitas e quantas foram para quarentena, com timestamp. Permite monitorar a qualidade dos dados por fonte de forma agregada, sem precisar abrir o arquivo detalhado de quarentena.

In [ ]:
# Log de processamento
log = pd.read_csv(LOG_PATH)
print(f"=== Log de Processamento ({len(log)} registros) ===")
print(log.to_string(index=False))

# Resumo de quarentena
print("\n=== Resumo de Quarentena ===")
print(pd.read_csv(QUARENTENA_RESUMO_PATH).to_string(index=False))

# Detalhes de quarentena (se houver)
q = pd.read_csv(QUARENTENA_LINHAS_PATH)
if len(q) > 0:
    print(f"\n=== Linhas em Quarentena ({len(q)} linhas) ===")
    print(q.head(10).to_string(index=False))
else:
    print("\nNenhuma linha em quarentena — todos os dados são válidos.")

## 8. Estrutura Final da Camada de Lote

Ao final da execução, a estrutura de diretórios gerada é:

```
projeto/
├── dados_brutos/
│   ├── 2022/
│   │   ├── 01/dados.csv
│   │   ├── 02/dados.csv
│   │   └── ... (12 meses)
│   └── 2023/
│       └── ... (12 meses)
├── gerenciamento/
│   ├── log_processamento.csv
│   ├── quarentena_linhas_brutas.csv
│   └── quarentena_linhas_por_arquivo.csv
└── vistas_lote/
    ├── receita_por_mes.csv
    ├── vendas_por_categoria.csv
    ├── vendas_por_regiao.csv
    ├── vendas_por_pagamento.csv
    ├── desconto_por_categoria.csv
    └── vendas_por_dia.csv
```

In [ ]:
# Exibe a estrutura de diretórios gerada
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in sorted(dirs) if not d.startswith('.')]
    level   = root.replace(".", "").count(os.sep)
    indent  = "│   " * level + "├── "
    folder  = os.path.basename(root)
    if folder in ("dados_brutos", "vistas_lote", "gerenciamento") or \
       any(folder in root for folder in ["dados_brutos", "vistas_lote", "gerenciamento"]):
        print(f"{indent}{os.path.basename(root)}/")
        for f in sorted(files):
            sub = "│   " * (level + 1) + "├── "
            print(f"{sub}{f}")